# 11 · Summary statistics from skewers through TNG sims

The summary statistics of notebook 10, measured on skewers through the TNG sims instead of the toy boxes:
the density on the skewers, the transmission and the overflux, and the 3D, isotropic, wedge, P1D, $\xi_{\rm 1D}$ and
decomposition statistics, with the same functions and figures. The simulation path (the snapshot, its cells and the
deposition) is explained in notebooks 05 and 07, the statistics of the cells of the same snapshot in notebook 09; the
statistics on skewers, their bins and conventions in notebook 10. Four lines (`LINE_IDS`: $\mathrm{Ly}\alpha$, Si III 1206, Si II 1193,
Si II 1190), and the BAO scale $r_d$ of the simulation's cosmology, which a box of $L = 51.7$ cMpc sees only through its
periodic images.

- **Part A — skewers:** a regular lattice of skewers along one axis of the box (`AXIS`), periodic, in the snapshot frame.
- **Part C — transmission and overflux.**
- **Part D — summary statistics of the overfluxes:** D1 bins and statistics, D2 planes, D3 $P$, $r^2\xi$ and wedges,
  D4 P1D next to DESI DR1, D5 by order, D6 term by term.

One simulation has no independent realisations, and the skewers run along one axis, `AXIS` ($y$ by default, as the
skewers through the toy boxes of 10), fixed with their direction: the ensemble has one member, as 10 with one box
(`n_real=1`), and the statistics have no error bars (the error of the mean needs two members).

In [ ]:
import os
os.environ.setdefault("JAX_PLATFORMS", "cpu")
import logging; logging.getLogger("jax._src.xla_bridge").setLevel(logging.CRITICAL)
import time, warnings
import numpy as np, jax, jax.numpy as jnp
# float32, the package's precision; skewerjax.config.set_precision("float64") for float64 (docs/precision.md)
import matplotlib.pyplot as plt
from skewerjax.utils.plotting import use_style, use_backend
use_backend()                                    # interactive figures in a live notebook
use_style()
from skewerjax.utils.plotting import add_panel_letters, plot_dag
from skewerjax.core import LineBundle, phys
from skewerjax.utils.paths import DATA_DIR as DATA      # folder with simulation data: $SKEWERJAX_DATA (docs/installation.md)
plot_dag(("lines", "skewers", "simulation", "fields", "profiles", "deposition", "tau", "normalisation", "overflux", "stats_1d",
          "stats_3d")); plt.show()

In [ ]:
from skewerjax.simulations import ArepoHDF5Reader
from skewerjax.core import sound_horizon_drag
from skewerjax.simulations import TNG_COSMOLOGY as TNG
from skewerjax.utils.plotting import fold_periodic
reader = ArepoHDF5Reader(f"{DATA}/TNG100-3", 25)          # the snapshot of notebooks 05 and 07
H = reader.header
L = H.box_cMpc                                           # the box [cMpc]
Z = H.redshift                                           # the redshift of the snapshot
u = H.velfac_kms_per_cMpc                                # aH [km/s per cMpc]: v = u x
R_BAO = float(sound_horizon_drag(TNG["Omega_b"] * H.h**2, H.omega_m * H.h**2))  # r_d [Mpc] (Aubourg et al. 2015, eq. 16): the BAO scale, comoving
print(f"z = {Z:.4f}, a = {H.a:.5f}, box L = {L:.3f} cMpc, {H.n_gas:,} gas cells")
print(f"u = aH = {u:.3f} km/s per cMpc, v_per = u L = {u * L:.1f} km/s (H.vmax_kms = {H.vmax_kms:.1f})")
n_img = np.arange(-4, 5)
print(f"r_BAO = r_d = {R_BAO:.2f} cMpc = {R_BAO / L:.3f} L (L = {L:.2f} cMpc): on the line of sight its images sit at |r_d - n L| folded into "
      f"[0, L / 2]: {np.unique(np.round(fold_periodic(R_BAO - n_img * L, L), 2))} cMpc (notebook 09)")

In [ ]:
# The skewers and their statistics: the same names as in notebook 10
AXIS = 2                           # the line of sight of the skewers: 1, 2, 3 = x, y, z (y: as the toy boxes of 10); one member
LINE_IDS = ("HI-LyA", "SiIII-1206")   # "SiII-1193", "SiII-1190"
LINES = LineBundle(list(LINE_IDS), ref_line_id=LINE_IDS[0])   # line 1, the reference: Lyα
N_SK = 64                          # skewers per transverse axis: N_SK^2 per line of sight, L / N_SK apart
SPACING_KMS = 1.0                  # a point of the paths every SPACING_KMS km/s of Hubble flow (resolution check, Part A)
N_PIX, BATCH = 128, 256            # velocity pixels over one period; skewers per call (memory only)
DETUNING = "linear"                # as 10: the first-order frequency offset of the profiles (changes tau by < 1e-5)
HUBBLE_FLOW = "linear"             # the snapshot frame is linear by construction ((aH)_snap y): "exact" gives the same line centres
A_EFF = None                       # not needed: in the snapshot frame every piece is at the snapshot's a (a_eff = a changes nothing)
MEAN = "redshift"                  # T_bar(z); one snapshot: every pixel at the same a, so the global mean (checked in Part C)

In [ ]:
MU_EDGES_XI = (0.0, 0.5, 0.8, 0.95, 1.0)   # the wedges of xi in mu = r_par / r (those of DESI DR2 Lya BAO, MU_WEDGES)
MU_EDGES_P = np.sin(np.linspace(0.0, np.pi / 2, 17))   # the 16 wedges of P in mu = k_par / |k|, evenly spaced in the angle to the line of sight: a separate input
ENSEMBLE = f"1 line of sight ({'xyz'[AXIS - 1]})"
EACH = "each line of sight ({n})"   # the legend of the faint curves ({n}: their number)

## Part A — Skewers

### A1 · Skewers through the box, along one axis

Along the axis `AXIS`, $N_{\rm SK}^2$ straight skewers through the whole box on a regular transverse lattice, $L/N_{\rm SK}$
apart (the nodes $i\,L/N_{\rm SK}$ of the two other axes), from 0 to $L$: the first and last points are the same place
of the periodic box, so every skewer is periodic along the line of sight and the samples form the lattice of 10's
3D statistics. `SimulationSkewers.axis_aligned` gives the points and the snapshot frame; every cell of the snapshot is
read once (`select_cells_in_region`, `gas_to_cells`, the default `ExtractionOptions` of 05 and 07: H I stored, Si III
and Si II from CLOUDY), and `cell_pieces` and `compute_optical_depth` run as in 10.

**The set-up** (the parameters cell, with the names of notebook 10): `N_SK` skewers per transverse axis, a point of the paths every `SPACING_KMS` km/s of Hubble flow and `N_PIX` velocity pixels over one period. The spacing is set directly here: the Voronoi cells have no common size, so there is no `POINTS_PER_CELL` (10 derives it from `SPACING_KMS`).

In [ ]:
from tqdm.auto import tqdm
from skewerjax.core import compute_optical_depth, lattice_columns, velocity_span
from skewerjax.core.absorbers import cell_pieces
from skewerjax.simulations import SimulationSkewers, gas_to_cells, select_cells_in_region
from skewerjax.summary_stats import lattice_limits, on_lattice

In [ ]:
t0 = time.perf_counter()
GAS = select_cells_in_region(reader, 0.0, L)                   # every cell of the snapshot (GasCells), read once
CELLS = gas_to_cells(GAS, LINES.species_keys)                  # -> core.Cells: per species n (comoving), T, v
t_read = time.perf_counter() - t0
COLS = lattice_columns(N_SK, (1, 1))                           # (N_SK^2, 2) the transverse indices (i_a, i_b) of the skewers, a-major
D_PERP = L / N_SK                                              # the transverse spacing [cMpc]
TRANSVERSE = {1: (1, 2), 2: (0, 2), 3: (0, 1)}                 # the transverse axes (a, b) of each line of sight


def cofm(axis, cols):
    # a point of each skewer [code units]: the transverse nodes (i_a, i_b) D_PERP
    p = np.zeros((len(cols), 3))
    a, b = TRANSVERSE[axis]
    p[:, a], p[:, b] = cols[:, 0] * D_PERP, cols[:, 1] * D_PERP
    return p / H.code_length_cMpc


SK0 = SimulationSkewers.axis_aligned(reader, cofm(AXIS, COLS[:1]), [AXIS], SPACING_KMS, cells=GAS)   # one skewer: the geometry
FRAME = SK0.frame("snapshot")                                  # the snapshot frame: one a, linear Hubble flow
PERIOD = float(velocity_span(FRAME, HUBBLE_FLOW)[0])           # v_per [km/s]: (aH)_snap L
N_LOS = SK0.points.shape[1] - 1                                # points along a skewer, the end point (= the first) dropped
DS = L / N_LOS                                                 # their spacing [cMpc]
V = (np.arange(N_PIX) + 0.5) * PERIOD / N_PIX                  # the velocity grid: N_PIX pixel centres over the period (Lyα axis)
DV = PERIOD / N_PIX
lim = lattice_limits(L, D_PERP, N_PIX, PERIOD)                 # the lattice of Part D: exactly the skewers' samples
print(f"skewers: {N_SK} x {N_SK} = {len(COLS)} along " + 'xyz'[AXIS - 1] + f", {D_PERP:.4f} cMpc apart ({ENSEMBLE}), in batches of {BATCH}")
print(f"points: {N_LOS} per skewer, every {DS * 1e3:.2f} ckpc = {DS * u:.3f} km/s; v_per = {PERIOD:.2f} km/s (= u L: {abs(PERIOD / (u * L) - 1):.0e}); "
      f"pixels: {N_PIX} of dv = {DV:.2f} km/s, P1D up to pi / dv = {np.pi / DV:.4f} s/km")
print(f"{len(CELLS):,} cells read and turned into core.Cells in {t_read:.1f} s; sizes V^(1/3) from {CELLS.size_cMpc.min():.3f} "
      f"to {CELLS.size_cMpc.max():.2f} cMpc (median {np.median(CELLS.size_cMpc):.3f})")

**$\tau$ along `AXIS`.** For every batch of skewers: the pieces of the cells (`cell_pieces`) and $\tau$ of the
lines of `LINE_IDS` on the $\mathrm{Ly}\alpha$ axis (`compute_optical_depth`, every option in the call), and the density at the points
(the column of each point's dual segment over its length, the end point merged with the first).

In [ ]:
TAU, DENS, RES, TIMES = [], [], [], []      # tau (N_lines, N_sk, N_PIX) of the one member (AXIS), the density at the points, resolution checks, timings
t0 = time.perf_counter()
bar = tqdm(total=len(COLS), desc="skewers")
with warnings.catch_warnings(record=True) as WARN:                     # the resolution check of compute_optical_depth (counted below)
    warnings.simplefilter("always")
    t_p = t_t = 0.0
    tau_a, dens_a = [], {sp: [] for sp in LINES.species_keys}
    for b in range(0, len(COLS), BATCH):
        t1 = time.perf_counter()
        cols_b = COLS[b:b + BATCH]
        sk = SimulationSkewers.axis_aligned(reader, cofm(AXIS, cols_b), np.full(len(cols_b), AXIS), SPACING_KMS, cells=GAS)
        pieces, parts = cell_pieces(CELLS, LINES, points=sk.points, frame=sk.frame("snapshot"), a_data=H.a, return_parts=True,
                                    deposition="voronoi_segments")    # cells -> pieces; the dual segments: the densities at the points need them
        seg, near = parts["segments"], parts["cells"]                # the dual segments and the cells they index
        for sp, d in dens_a.items():                                 # sum_p n_p l_p over each dual segment (SimulationSkewers.point_values)
            n_c = np.asarray(near.species[sp]["number_density"])
            col = np.where(seg.cell >= 0, seg.weight * n_c[np.maximum(seg.cell, 0)] * seg.length, 0.0).sum(-1)
            col[:, 0] += col[:, -1]                                  # the first and last points: one dual segment
            d.append(col[:, :-1] / DS)                               # its mean density [comoving cm^-3]
        t2 = time.perf_counter()
        tau_b, info = compute_optical_depth(  # CALCULATION OF THE OPTICAL DEPTH tau
            pieces, V, LINES, line_frame="reference", velocity_period=True, detuning=DETUNING,
            hubble_flow=HUBBLE_FLOW, aH_ref=None, a_eff=A_EFF, resolution_tolerance=0.25, return_info=True
        )
        tau_a.append(np.asarray(tau_b))
        RES.append(np.stack([info["resolution"]["max_ratio"], info["resolution"]["column_fraction"]], -1))
        t_p, t_t = t_p + t2 - t1, t_t + time.perf_counter() - t2
        bar.update(len(cols_b))
    TAU.append(np.concatenate(tau_a, axis=1))
    DENS.append({sp: on_lattice(np.concatenate(d), N_SK, N_SK) for sp, d in dens_a.items()})   # (N_SK, N_LOS, N_SK)
    TIMES.append((t_p, t_t))
bar.close()
TAU, TIMES, RES = np.stack(TAU), np.array(TIMES), np.array(RES)  # TAU (n_ens = 1, N_lines, N_sk, N_PIX): one member, as 10 with one box
t_skewers = time.perf_counter() - t0
del GAS, SK0, sk, pieces, parts, seg, near

In [ ]:
from skewerjax.utils.plotting import line_names
NAMES_L = line_names(LINE_IDS)                                   # Lyα, Si III, Si II 1193, Si II 1190
n_ens, n_sk = TAU.shape[0], TAU.shape[2]                         # the members of the ensemble (one line of sight) and the skewers of each
T_REF = 1e4                                                      # the temperature of the thermal widths b [K]
b_th = {lid: np.sqrt(2 * phys.kB * T_REF / m) / 1e3 for lid, m in zip(LINE_IDS, np.asarray(LINES.mass_amu))}
print(f"{n_ens} line of sight ({ENSEMBLE}) x {n_sk} skewers in {t_skewers:.0f} s: pieces {TIMES[:, 0].mean():.0f} s, tau {TIMES[:, 1].mean():.0f} s")
print(f"resolution check (W <= b/4): max W / b = {np.round(RES[..., 0].max(axis=0), 3)}, column fraction of the pieces wider than b/4 "
      f"at most {np.round(RES[..., 1].max(axis=0), 6)} ({', '.join(NAMES_L)}); warnings: {len({str(w.message) for w in WARN})} distinct, "
      f"{len(WARN)} in all; b at {T_REF:g} K: " + ", ".join(f"{lid} {b:.2f} km/s" for lid, b in b_th.items()))
print("density at the points [comoving cm^-3]: " + ", ".join(f"<n_{sp}> = {np.mean([d[sp].mean() for d in DENS]):.3e}" for sp in LINES.species_keys)
      + "; the cells' volume means: " + ", ".join(
      f"{sp} {np.sum(CELLS.species[sp]['number_density'] * CELLS.volumes) / L**3:.3e}" for sp in LINES.species_keys))

## Part C — Transmission and overflux

$T = e^{-\tau}$, its mean $\bar T$ over the pixels of `MEAN` and $\delta = T/\bar T - 1$ (10, Part C). In one snapshot
every pixel has the same scale factor, so $\bar T(z)$ is one number, the global mean.

In [ ]:
from skewerjax.summary_stats import ensemble_mean_transmissions, overflux_check
A_PIX = np.full(N_PIX, H.a)                                                     # the scale factor of every pixel: one snapshot
T = np.exp(-TAU)                                                                # the transmission of each line (n_ens, N_lines, N_sk, N_PIX)
T_BAR = ensemble_mean_transmissions(T, mean=MEAN, grid=A_PIX)                   # T_bar of every line and of the total: (n_ens, N_lines + 1, N_sk, N_PIX), a view
DELTA = np.concatenate([T, T.prod(axis=1, keepdims=True)], axis=1) / T_BAR - 1  # delta = T / T_bar - 1 (every line, total)
CHECK = overflux_check(DELTA, T_BAR, mean=MEAN, grid=A_PIX)                     # <delta> = 0 over the sets of MEAN, C, the decomposition
T_glob = ensemble_mean_transmissions(T, mean="global")
T_v = ensemble_mean_transmissions(T, mean="redshift")[0, :, 0]                  # without a grid: the mean at each pixel (a diagnostic)
print("mean transmission: " + ", ".join(f"{nm} {T_BAR[0, j, 0, 0]:.6f}" for j, nm in enumerate([*NAMES_L, "total"])) + f" ({ENSEMBLE})")
print(f"MEAN = {MEAN!r} on one snapshot (every pixel at a = {H.a:.5f}): the global mean, max |T_bar / T_bar_global - 1| = "
      f"{np.max(np.abs(T_BAR[:, :, 0] / T_glob[:, :, 0] - 1)):.1e}; the mean at each pixel instead would scatter by "
      f"{np.round((T_v / T_v.mean(axis=1, keepdims=True)).std(axis=1), 4)} (rms, every line and the total)")
print(f"normalisation: max |<delta>| over the sets of MEAN = {CHECK['mean_delta_level'].max():.1e}; C = T_bar / prod_j T_bar_j - 1 = "
      f"{CHECK['C']:.4e}, |C - sum_(|A| >= 2) <delta_A>| = {CHECK['C_error']:.1e}; max |[sum_A delta_A - C] / (1 + C) - delta_tot| = "
      f"{CHECK['decomposition_error']:.1e}")
del T

### C1 · One skewer, from the gas to the flux contrast

In [ ]:
from skewerjax.simulations import voronoi_slice
from skewerjax.utils.plotting import plot_skewer_slice, plot_toy_skewer
i0 = 0                                                                    # the member of the ensemble shown: the one line of sight
a_ax, b_ax = TRANSVERSE[AXIS]                                             # the transverse axes (0-based): across in the map, the normal of the plane
on_b0 = np.flatnonzero(COLS[:, 1] == 0)
j = int(on_b0[np.argmin(np.abs(COLS[on_b0, 0] - N_SK // 2))])            # the skewer nearest to L/2 across, in the layer i_b = 0
ix, iz = COLS[j]
SLICE = voronoi_slice(CELLS, "HI", b_ax + 1, iz * D_PERP, n_pix=512)      # the Voronoi cells cut by the plane of that layer: n_HI
n_map = SLICE["n"] if AXIS == max(SLICE["axes"]) else SLICE["n"].T       # [across, along the line of sight], as the toy fields of 10
fig, ax, sl_info = plot_skewer_slice(n_map, L, COLS, z_index=0, highlight=(ix, iz), spacing=D_PERP)
plt.show()
print(f"{sl_info['n_row']} skewers in the plane " + "xyz"[b_ax] + f" = {iz * D_PERP:.1f} cMpc (line of sight " + "xyz"[AXIS - 1] + ", across "
      + "xyz"[a_ax] + f"), {sl_info['spacing_cMpc']:.4g} cMpc apart; the skewer of the next figure: " + "xyz"[a_ax] + f" = {ix * D_PERP:.2f} cMpc; "
      f"the map: the n_HI of the Voronoi cell at every pixel centre ({SLICE['n_cells']:,} cells searched)")

The metal lines absorb little in this snapshot (their $\bar T$ in Part C): the total lies on the $\mathrm{Ly}\alpha$ curve.

In [ ]:
from skewerjax.utils.plotting import line_pairs
n_skewer = {sp: DENS[i0][sp][ix, :, iz] for sp in LINES.species_keys}       # the density at the points of this skewer
fig, axs, grid_info = plot_toy_skewer(L, n_skewer, V, TAU[i0][:, j], mean_transmission=T_BAR[i0][:, j], line_ids=LINE_IDS)
add_panel_letters(axs)
plt.show()
PAIRS = line_pairs(LINE_IDS, np.asarray(LINES.delta_v), PERIOD)           # every pair of lines, its separation folded into [0, v_per / 2]
dv_12 = float(LINES.delta_v[1])
dv_fold = PAIRS[0]["fold"]                                                # Si III - Lya, folded
print(f"line separations Delta v [km/s], folded into [0, v_per / 2] with v_per = {PERIOD:.1f} km/s [km/s and cMpc]: " + "; ".join(
      f"{NAMES_L[p['i']]}–{NAMES_L[p['j']]} {p['dv']:.1f} -> {p['fold']:.1f} = {p['fold'] / u:.2f}" for p in PAIRS))
print(f"b(Lya) / b(Si) = {b_th[LINE_IDS[0]] / b_th[LINE_IDS[1]]:.2f}")

## Part D — Summary statistics of the overfluxes

As 10's Part D: along the line of sight a velocity $v$ [km/s], across it comoving distances [cMpc], related by
$u = aH$ (exact here: one snapshot); the statistics on exactly the lattice of the skewers. One line of sight, one
member (as 10 with one box): no error bars.

### D1 · The bins and the statistics

The bins of 10 (D1): $r_\perp$ `RPERP_EDGES_C` (`set_rperp_edges`: $r_\perp = 0$ alone, one bin per skewer separation to
$r_1$, then steps of $s\,dx$), $\lvert\mathbf r\rvert$ `R_EDGES_C` (`set_r_edges`: one bin per separation to $r_1$, the zero
lag in none, then steps of the coarser spacing), $\lvert\mathbf k\rvert$ `K_EDGES_C` (`set_k_edges`).

In [ ]:
from skewerjax.summary_stats import lattice_grid, set_k_edges, set_r_edges, set_rperp_edges
# The bins of the 3D statistics of the overfluxes (D1) in comoving units [cMpc], [cMpc^-1] (the line of sight converted with u)
dv_c = DV / u                                                                   # a pixel dv / u [cMpc]
k_nyq_c = min(np.pi / D_PERP, np.pi / dv_c)                                     # the smaller Nyquist frequency
RPERP_EDGES_C, RPERP_INFO_C = set_rperp_edges(D_PERP, L=L, r_1=3 * D_PERP, r_2=L / 2, r_max=L / 2, dr_factor=1.0, n_log=0, merge_empty=True, info=True)   # r_perp: one bin per skewer separation to r_1 (the first: the same skewer), then steps of d_perp to L / 2
R_EDGES_C, R_INFO_C = set_r_edges(D_PERP, dv_c, L=L, r_1=2 * D_PERP, r_2=L / 2, r_max=L / 2, dr_factor=1.0, n_log=0, merge_empty=True, info=True)   # |r|: one bin per separation to r_1, then steps of max(d_perp, dv / u) to L / 2
K_EDGES_C = set_k_edges(2 * np.pi / L, k_1=3.5 * 2 * np.pi / L, k_2=k_nyq_c, k_max=k_nyq_c, dk_factor=0.4)   # |k|: one bin per mode, then steps of 0.4 k_f to the smaller Nyquist frequency
EDGES_C = dict(k_edges=K_EDGES_C, r_edges=R_EDGES_C, rperp_edges=RPERP_EDGES_C)
print(f"overflux bins: r_perp {RPERP_EDGES_C.size - 1} ({RPERP_INFO_C['n_sep']} one per skewer separation to {RPERP_INFO_C['r_1']:.3f} cMpc, the first "
      f"r_perp = 0; {RPERP_INFO_C['n_step']} steps of {RPERP_INFO_C['dr']:.4g} cMpc to {RPERP_EDGES_C[-1]:.1f} cMpc); r_par one row per pixel separation, "
      f"{N_PIX // 2 + 1} rows from 0 to v_per / 2; |r| {R_EDGES_C.size - 1} ({R_INFO_C['n_sep']} one per separation, from {R_EDGES_C[0]:.4f} to "
      f"{R_INFO_C['r_1']:.4f} cMpc; {R_INFO_C['n_step']} steps of {R_INFO_C['dr']:.4g} cMpc to {R_EDGES_C[-1]:.1f} cMpc; separation vectors per bin "
      f"{R_INFO_C['n_lags'].min()} to {R_INFO_C['n_lags'].max()}, {R_INFO_C['n_merged']} empty bin(s) merged); |k| {K_EDGES_C.size - 1} up to {K_EDGES_C[-1]:.4f} cMpc^-1")
print(f"full coverage of the lattice: |k| <= min(pi / d_perp, pi u / dv) = min({np.pi / D_PERP:.4f}, {np.pi / dv_c:.4f}) cMpc^-1, "
      f"|r| <= L / 2 = {L / 2:.1f} cMpc")

In [ ]:
from skewerjax.summary_stats import GRID_KEYS, ensemble_mean, measure_lattice, skewer_p1d, skewer_xi1d
OVERFLUX_FIELDS = (*LINE_IDS, "total")                          # the overflux of each line and of the total
GRID_C = lattice_grid(N_SK, N_SK, N_PIX, L, D_PERP, **EDGES_C, xi_mu_edges=MU_EDGES_XI, pk_mu_edges=MU_EDGES_P,
                      log=print)                                # the 3D statistics on the lattice, bins EDGES_C and the wedges
P1D_EACH, XI1D_EACH, F3D_EACH = [], [], []
for i in range(n_ens):                                          # every statistic per member of the ensemble, from arrays and geometry only
    K1D, p = skewer_p1d(DELTA[i], DV)                           # P1D (N_lines + 1, N_PIX / 2 + 1): one 1D FFT per skewer [km/s]
    LAGS, x = skewer_xi1d(DELTA[i], DV)                         # xi1D (N_lines + 1, N_PIX / 2 + 1) at every lag j dv
    P1D_EACH.append(dict(p1d=p))
    XI1D_EACH.append(dict(xi1d=x))
    # skewer samples: on the skewers' own lattice, along the line of sight N_PIX pixels, across it the skewer spacing; no deposition, window or interlacing
    F3D_EACH.append({f: measure_lattice(DELTA[i, jf], GRID_C) for jf, f in enumerate(OVERFLUX_FIELDS)})  # P, xi in 3D
P1D = dict(ensemble_mean(P1D_EACH, ("p1d",)), k=K1D)            # the mean over the ensemble and its error, k [s/km]
XI1D = dict(ensemble_mean(XI1D_EACH, ("xi1d",)), lags=LAGS)
FLUX3D = {f: ensemble_mean([r[f] for r in F3D_EACH], GRID_KEYS) for f in OVERFLUX_FIELDS}
AXES3D = GRID_C.axes()                                          # the bins of D1 [cMpc], [cMpc^-1]
print(f"statistics of {ENSEMBLE}: P1D and xi1D of {len(OVERFLUX_FIELDS)} overfluxes along {n_sk} skewers each, the 3D statistics on "
      f"the lattice {GRID_C.shape} (overflux)")

### D2 · The $(k_\perp, k_\parallel)$ & $(r_\perp, r_\parallel)$ planes

As in 10 (D2): both maps start at the origin; `show_mu`, `show_contours`, `show_modes` and `show_pairs` are written in each
call, and the dots mark where the value of each cell is computed (the mode- or pair-weighted mean of the cell).

In [ ]:
from skewerjax.utils.plotting import plot_overflux_kpar_kperp, plot_overflux_rpar_rperp
from skewerjax.utils.plotting import field_names, fold_periodic
ax3, f3 = AXES3D, FLUX3D                                                 # comoving bins [cMpc], [cMpc^-1]
NAMES = field_names(OVERFLUX_FIELDS)                                     # Lyα, Si III, Si II 1193, Si II 1190, total
tot = dict(f3["total"], axes=ax3)                                        # the total overflux and its bins
fig, ax, _ = plot_overflux_kpar_kperp(tot, kms_per_cMpc=u, kperp_edges=K_EDGES_C, n_sigma=1, kpar_max=k_nyq_c,  # drawn: k_par <= pi / d_perp (square)
                                      show_mu=True, show_contours=True, show_modes=True)   # the dots: where each cell's P is computed
plt.show()
print(f"k map: {ax3['kpar_edges'].size - 1} rows, k_par = 2 pi m / v_per from {2 * np.pi / PERIOD:.2e} to pi / dv = {np.pi / DV:.2e} s/km; "
      f"{ax3['kperp_edges'].size - 1} columns of k_perp (K_EDGES_C), of which {int(np.sum(ax3['n_modes_P_map'].sum(0) == 0))} hold no mode; "
      f"shown up to k_par = pi / d_perp = {k_nyq_c / u:.4f} s/km")

In [ ]:
fig, ax, curves = plot_overflux_rpar_rperp(tot, kms_per_cMpc=u, rperp_edges=RPERP_EDGES_C, r_bao=R_BAO, delta_v=np.asarray(LINES.delta_v), line_ids=LINE_IDS,
                                           pairs="all", n_sigma=1, color_scale="symlog", quantity="r2xi",
                                           show_mu=True, show_contours=True, show_pairs=True)  # the dots: where each cell's xi is computed; every pair, metal-metal included, in its colour: diamond, echo + dashed, echo - dotted; the ring grey dotted
plt.show()
SEP = dv_fold / u                                                        # the line separation in comoving units [cMpc], folded
print(f"map: {ax3['rpar_edges'].size - 1} rows, one per pixel separation j dv (dv = {DV:.2f} km/s) from 0 to {u * ax3['rpar'][-1]:.0f} km/s = v_per / 2; "
      f"the spots of the {len(PAIRS)} pairs (folded) at r_par = " + ", ".join(f"{p['fold']:.0f}" for p in PAIRS) + " km/s; r_BAO > L / 2: its images and "
      f"their echoes (n_perp = 0) as arcs")
X, Xe = f3["total"]["xi_map"], f3["total"]["xi_map_err"]
ip = int(np.argmin(np.abs(ax3["rpar"] - SEP)))                           # the r_par row of the spot
for q in (0, 1):                                    # r_perp columns: 0 = pairs on the same skewer, 1 = one transverse spacing
    excess = X[ip, q] - 0.5 * (X[ip - 2, q] + X[ip + 2, q])
    print(f"Si III - Lya at r_par = Delta, r_perp bin {q} ({ax3['rperp_edges'][q]:.2f}-{ax3['rperp_edges'][q + 1]:.2f} cMpc): xi = {X[ip, q]:.2e}, "
          f"excess over the rows two pixels away {excess:.2e} = {excess / Xe[ip, q]:.1f} sigma ({ENSEMBLE})")

**The echoes of the BAO ring.** As in 10 (D2): every pair of lines $(i, j)$ puts a spot on the line of sight at
$r_\parallel = \Delta v_{ij,\rm fold}/u$ (the diamond) and copies the BAO ring shifted by $\pm\Delta v_{ij}/u$ along it
(echo $+$ dashed, echo $-$ dotted), all folded with the period $L$. Here $r_{\rm BAO} = r_d > L/2$, so the ring and its
echoes are drawn as arcs of their periodic images about $(0, nL)$.

### D3 · Power spectrum, correlation function and wedges of the overfluxes

As in 10 (D3): shaded outside the limits of the lattice (`lattice_limits`, `coverage_limits`): $k_{\min} = 2\pi/L$,
$k_{\max}$ the smallest Nyquist frequency, $r_{\min} = \min(s\,dx, dv/u)$ the smallest separation (the first bin of
`R_EDGES_C` holds it alone), $r_{\max} = L/2$; the wedges of $\xi$ alike.

In [ ]:
from skewerjax.utils.plotting import plot_overflux_3d
from skewerjax.summary_stats import bao_maxima
K_BAO_MAX = 1.0                                                      # [cMpc^-1]: the maxima of j_0(k r_d) continue up to the Nyquist frequency; beyond this there is no wiggle to mark (Silk damping)
BAO_K = bao_maxima(K_BAO_MAX, r_bao=R_BAO)                           # the maxima of j_0(k r_d)
fig, axs = plot_overflux_3d(f3, ax3, kms_per_cMpc=u, r_bao=R_BAO, bao_k=BAO_K, delta_v=np.asarray(LINES.delta_v), line_ids=LINE_IDS, box=L,
                            names=NAMES, k_edges=K_EDGES_C, r_edges=R_EDGES_C, limits=lim, quantity="xi", y_scale="symlog", k_scale="linear",
                            each_label=EACH)
add_panel_letters([axs["P"], axs["xi"]])
plt.show()
PT = f3["total"]["P_iso"]
print(f"lattice {N_SK} x {N_PIX} x {N_SK} per line of sight, a cube of side L = {L:.2f} cMpc (v_per = {PERIOD:.0f} km/s along the line of sight)")
print(f"|k| from {ax3['k_edges'][0]:.3f} to k_N = {ax3['k_edges'][-1]:.3f} cMpc^-1 in {ax3['k_edges'].size - 1} bins: Nyquist pi N_pix / L = "
      f"{lim['k_nyq_par']:.3f} cMpc^-1 along the line of sight, pi / d_perp = {lim['k_nyq_perp'][0]:.3f} cMpc^-1 across it")
print(f"|r| from r_min = {lim['r_min']:.3f} cMpc (dv / u, the smallest separation) to r_max = L / 2 = {lim['r_max']:.1f} cMpc in "
      f"{ax3['r_edges'].size - 1} bins (R_EDGES_C), the first centred at {ax3['r_iso'][0]:.3f} cMpc")
print("P_line / P_tot at the lowest |k|: " + ", ".join(f"{f} {f3[f]['P_iso'][0] / PT[0]:.3f}" for f in LINE_IDS))

**Wedges of the total overflux.** As in 10 (D3): the vertical lines are line-of-sight positions only ($r_\perp = 0$,
$\mu = 1$), folded into $[0, L/2]$: $r_{\rm BAO}$ grey and dotted, and for every pair, in its colour, the spot
$\Delta v_{ij}/u$ solid and the crossings of its echoes, $\lvert r_{\rm BAO} \pm \Delta v_{ij}/u\rvert$, dashed and dotted.
Shaded as the isotropic figure; the points of every wedge at the bins of `R_EDGES_C`; a wedge with no separation in a bin
is NaN there: an open marker on the line between two computed bins, nothing at the ends.

In [ ]:
from skewerjax.utils.plotting import los_scales, plot_xi_mu_wedges
LOS = los_scales(R_BAO, LINE_IDS, np.asarray(LINES.delta_v) / u, L)  # r_BAO, the spot and the echoes of every pair on the line of sight [cMpc], folded
fig, axs, _ = plot_xi_mu_wedges(tot, r_edges=R_EDGES_C, r_bao=R_BAO, delta_v=np.asarray(LINES.delta_v), line_ids=LINE_IDS, box=L, kms_per_cMpc=u, quantity="r2xi", y_scale="linear",
                                limits=lim, each_label=EACH)   # shaded as D3
plt.show()
print(f"line-of-sight (mu = 1) scales [cMpc], folded into [0, L / 2]: r_BAO {LOS['ring']:.2f} (r_d = {R_BAO:.1f}); spot, |r_BAO + Delta|, |r_BAO - Delta| of every pair: " + "; ".join(
      f"{NAMES_L[p['i']]}–{NAMES_L[p['j']]} {p['spot']:.2f}, {p['plus']:.2f}, {p['minus']:.2f}" for p in LOS["pairs"]))
mu_e, r_mu = ax3["mu_edges"], ax3["r_mu"]
js = np.nanargmin(np.abs(r_mu - SEP), axis=1)
print(f"the Si III - Lya spot at r = {SEP:.2f} cMpc (mu = 1 wedge only); r^2 xi of the total in the |r| bin of the spot [cMpc^2]: " + "; ".join(
      f"{mu_e[w]:.2f} < mu < {mu_e[w + 1]:.2f}: {r_mu[w, jj]**2 * f3['total']['xi_mu'][w, jj]:+.3f} +- {r_mu[w, jj]**2 * f3['total']['xi_mu_err'][w, jj]:.3f}"
      for w, jj in enumerate(js)))

**Wedges of $P$ of the total overflux.** As in 10 (D3): $P$ in the wedges of `MU_EDGES_P`
($\mu = k_\parallel/\lvert\mathbf k\rvert$, $k_\parallel$ along the skewers in cMpc$^{-1}$ through $u$) on the bins
`K_EDGES_C`; one line of sight, so no error bars;
no expectation. Dotted grey: the maxima of $j_0(k r_d)$ with TNG's $r_d$; solid in the colour of each
pair, its line-of-sight ($\mu = 1$) imprint at $k = 2\pi n/\Delta_{ij}$, $\Delta_{ij} = \Delta v_{ij,\rm fold}/u$
folded with the period $L$; in a wedge of mean $\mu$ it sits at $2\pi n/(\mu\Delta_{ij})$. The points of every wedge sit
at the bins of `K_EDGES_C` (the mode-weighted $\lvert\mathbf k\rvert$ of each bin, as in the isotropic figure); a wedge with no
mode in a bin is NaN there: an open marker on the line between two computed bins, nothing at the ends. Unlike 10, the
narrowest wedge is not near zero at small $k$: in the snapshot frame every pixel has the same $a$, so `MEAN = "redshift"`
is one global mean and the plane $k_\perp = 0$ keeps its power.

In [ ]:
from skewerjax.summary_stats import pair_imprint
from skewerjax.utils.plotting import line_pairs, plot_pk_mu_wedges
fig, axs, _ = plot_pk_mu_wedges(tot, k_edges=K_EDGES_C, bao_k=BAO_K, delta_v=np.asarray(LINES.delta_v), line_ids=LINE_IDS, box=L, panels=True,
                                kms_per_cMpc=u, limits=lim, quantity="P", k_scale="linear", each_label=EACH)
add_panel_letters(axs)
plt.show()
for p in line_pairs(LINE_IDS, np.asarray(LINES.delta_v) / u, L):
    kk = pair_imprint(p["fold"], lim["k_max"], kind="p1d")             # k = 2 pi n / Delta_ij up to k_N
    print(f"{NAMES_L[p['i']]}–{NAMES_L[p['j']]}: Delta = {p['fold']:.2f} cMpc ({u * p['fold']:.0f} km/s), mu = 1 imprint at k = 2 pi n / Delta = "
          f"{np.round(kk, 3)} cMpc^-1 = {np.round(kk / u, 4)} s/km")
k_si = pair_imprint(SEP, lim["k_max"], kind="p1d")[0]   # the first Si III - Lya imprint [cMpc^-1]
mu_p, ke = ax3["pk_mu_edges"], ax3["k_mu_edges"]
jk = int(np.searchsorted(ke, k_si, side="right") - 1)                  # its |k| bin
PW, PWe = f3["total"]["P_mu"][:, jk], f3["total"]["P_mu_err"][:, jk]
print(f"P of the total in the |k| bin {ke[jk]:.4f}-{ke[jk + 1]:.4f} cMpc^-1 of the first Si III - Lya imprint (k = {k_si:.4f}) [cMpc^3]: " + "; ".join(
      f"{mu_p[w]:.2f} < mu < {mu_p[w + 1]:.2f}: {PW[w]:.4g} +- {PWe[w]:.2g}" for w in range(PW.size)))

**Reading.** On large scales $P$ rises towards the line of sight: at the first Si III–$\mathrm{Ly}\alpha$ imprint
($k \simeq 0.28$ cMpc$^{-1}$, printed) from $25 \pm 3$ cMpc$^3$ across it ($\mu < 0.5$) to $125 \pm 43$ along it
($\mu > 0.95$), the redshift-space distortions of the overflux, as in the redshift space of 09. The wedges meet above
$k \simeq 1.5$ cMpc$^{-1}$. The imprint of the pair at $k = 2\pi n/\Delta$, $\Delta = 22.1$ cMpc, does not stand out
against the errors of the three lines of sight: Si III absorbs little in this snapshot (Part C), so its cross term with
$\mathrm{Ly}\alpha$ is weak.

### D4 · P1D next to a DESI measurement

The DESI DR1 P1D at every redshift of Chaves-Montero et al. (Fig. 1), $z = 2.2$ to $4.2$, read from the vector paths of the
figure (`utils.reference_data`), one colour per $z$, small points joined by a thin line (the snapshot is at $z = 3.008$). The
simulation's mean transmission is not rescaled to the observed one. The dotted grey lines are the BAO maxima of the 3D spectrum at
$k_\parallel = k/u$, for reference: P1D integrates over $k_\perp$ and damps them. The solid lines, in the colour of each pair $(i, j)$, are the
imprint of its separation folded into the period, $k_\parallel = 2\pi n/\Delta v_{ij,\rm fold}$ (10, D4; the first `N_HARMONICS` of each pair). P1D is drawn on its native modes
$k_m = 2\pi m/v_{\rm per}$, $m = 1, \dots, N_{\rm pix}/2$, one point per mode (no rebinning), here and in D5 and D6.

In [ ]:
N_HARMONICS = 3                                                     # the first maxima of every pair's imprint on the P1D figures (all of them would be 100-200 lines)
from skewerjax.utils.plotting import plot_p1d_reference
from skewerjax.utils.reference_data import P1D_DESI_DR1, nearest_reference
P1D_REF = nearest_reference(P1D_DESI_DR1, Z)                            # the reference nearest to the snapshot, for the print (every z is drawn alike)
K_LIMITS_1D = (2 * np.pi / PERIOD, np.pi / DV)                                   # k_min, k_max of the line of sight [s/km]
BAO_K_1D = bao_maxima(K_BAO_MAX, r_bao=R_BAO, kms_per_cMpc=u)                     # k_par = k / u of the 3D maxima [s/km]
fig, ax = plot_p1d_reference(P1D, P1D_DESI_DR1, names=NAMES, k_max=P1D_REF["k_max_figure"], k_limits=K_LIMITS_1D,  # the native modes
                             kms_per_cMpc=u, each_label=EACH, bao_k=0, delta_v=np.asarray(LINES.delta_v), line_ids=LINE_IDS, n_harmonics=1)
plt.show()
kc = P1D["k"]
sim = np.interp(P1D_REF["k"], kc, kc * P1D["p1d"][0] / np.pi)
print(f"snapshot z = {Z:.3f}, the {len(P1D_DESI_DR1)} reference redshifts {min(P1D_DESI_DR1):g} to {max(P1D_DESI_DR1):g}, the nearest z = {P1D_REF['z']:g}; reference / simulation (Lya) over its k range: median "
      f"{np.median(P1D_REF['kP_pi'] / sim):.2f}, from {np.min(P1D_REF['kP_pi'] / sim):.2f} to {np.max(P1D_REF['kP_pi'] / sim):.2f}")
print("line separations: periods 2 pi / Delta v [s/km]: " + ", ".join(f"{NAMES_L[p['i']]}–{NAMES_L[p['j']]} {2 * np.pi / p['dv']:.4f}" for p in PAIRS) + "; k_min = 2 pi / v_per = "
      f"{2 * np.pi / PERIOD:.2e}, k_max = pi / dv = {np.pi / DV:.3f} s/km; P1D on its {kc.size - 1} native modes (no rebinning)")

### D5 · The decomposition by order

`decompose_1d` (opt-in, as in 10, D5): P1D and $\xi_{\rm 1D}$ of the total split into the pairs of subset terms and their
orders.

In [ ]:
from skewerjax.summary_stats import decompose_1d, ensemble_1d
from skewerjax.utils.plotting import plot_p1d_orders, plot_xi1d_orders
FLUX_ORDERS = ensemble_1d([decompose_1d(TAU[i], DV, T_BAR[i], mean=MEAN) for i in range(n_ens)])  # opt-in
dv_lines = np.asarray(LINES.delta_v)                                             # the offsets of every line: every pair's separation
fig, axs = plot_p1d_orders(FLUX_ORDERS, k_limits=K_LIMITS_1D, kms_per_cMpc=u, each_label=EACH, delta_v_list=dv_lines, line_ids=LINE_IDS, n_harmonics=N_HARMONICS, k_scale="linear")
add_panel_letters(axs)
plt.show()
fig, axs = plot_xi1d_orders(FLUX_ORDERS, delta_v_list=dv_lines, line_ids=LINE_IDS, kms_per_cMpc=u, each_label=EACH, quantity="xi", y_scale="symlog")  # every lag, 0 .. v_per / 2
add_panel_letters(axs)
plt.show()
e = FLUX_ORDERS
print(f"max |sum of the orders - total| / max |total|: P {np.abs(e['P_order'].sum(0) - e['P_total']).max() / np.abs(e['P_total']).max():.1e}, "
      f"xi {np.abs(e['xi_order'].sum(0) - e['xi_total']).max() / np.abs(e['xi_total']).max():.1e}; order 1 / max |total|: "
      f"P {np.abs(e['P_order'][1]).max() / np.abs(e['P_total']).max():.1e}")
xi_c, xi_d = e["xi_total"][N_PIX // 2:], XI1D["xi1d"][-1][: N_PIX // 2]  # the lags 0 .. v_per / 2 - dv (centred lags: 0 at N_PIX / 2)
print(f"the total of the decomposition against the per-skewer statistics of D1: P1D {np.abs(e['P_total'] - P1D['p1d'][-1]).max() / P1D['p1d'][-1].max():.1e}, "
      f"xi1D {np.abs(xi_c - xi_d).max() / np.abs(xi_d).max():.1e} (relative)")
print("orders: " + ", ".join(str(n) for n in range(e["P_order"].shape[0])) + f" ({len(LINE_IDS)} lines: 0 .. 2 N_lines); cos(k Delta v) of Si III - Lya: period "
      f"{2 * np.pi / abs(dv_12):.2e} s/km = {PERIOD / abs(dv_12):.2f} mode spacings (2 pi / v_per)")

### D6 · Each order, term by term

In [ ]:
from skewerjax.utils.plotting import plot_p1d_terms, plot_xi1d_terms
fig, axs = plot_p1d_terms(FLUX_ORDERS, k_limits=K_LIMITS_1D, kms_per_cMpc=u, line_ids=LINE_IDS, delta_v_list=dv_lines, n_harmonics=N_HARMONICS, k_scale="linear")  # the largest terms of each order
add_panel_letters(axs)
plt.show()
fig, axs = plot_xi1d_terms(FLUX_ORDERS, delta_v_list=dv_lines, kms_per_cMpc=u, line_ids=LINE_IDS)  # the separations, folded: solid, in the pairs' colours
add_panel_letters(axs)
plt.show()